# Molekül-Viewer (Playground)

Interaktive Visualisierung einzelner Molekül-Graphen aus einer `.pkl`-Datei (Standard: `all_graphs_with_length.pkl`). Optional können Node-/Edge-Features, Modell-Predictions sowie Importances aus `gnnexplainer.py` eingeblendet werden.

In [ ]:
import sys
import json
import pickle
from pathlib import Path
from functools import lru_cache
from typing import Any, Dict, Optional

import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
EXPLAIN_DIR = PROJECT_ROOT / "results" / "explanations"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
EXPLAINER_DIR = SCRIPTS_DIR / "explainer"

for path in [SCRIPTS_DIR, EXPLAINER_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

if TORCH_AVAILABLE:
    try:
        from explainer.explainer_utils import (
            build_dataset,
            load_config,
            load_stats,
            load_trained_model,
            heterodata_to_dicts,
        )
    except Exception as exc:  # noqa: BLE001
        print(f"Warnung: Konnte explainer_utils nicht laden ({exc}). Predictions/Importances sind deaktiviert.")
        build_dataset = load_config = load_stats = load_trained_model = heterodata_to_dicts = None
else:
    print("PyTorch nicht gefunden. Predictions und GNNExplainer-Importances sind deaktiviert.")
    build_dataset = load_config = load_stats = load_trained_model = heterodata_to_dicts = None

plt.rcParams["figure.dpi"] = 120


In [ ]:
def node_type_for(node_attrs: Dict[str, Any]) -> str:
    ntype = node_attrs.get("element")
    return ntype if ntype in ("H", "C") else "Others"


def list_pickles(directory: Path = DATA_DIR):
    if not directory.exists():
        return []
    return sorted([p.name for p in directory.glob("*.pkl")])


@lru_cache(maxsize=2)
def load_graphs_from_pickle(pkl_path: str):
    path = Path(pkl_path)
    if not path.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")
    with open(path, "rb") as handle:
        graphs = pickle.load(handle)
    if not isinstance(graphs, list):
        raise ValueError(f"Erwarte Liste von networkx-Graphen, erhielt {type(graphs)}.")
    return graphs


def collect_node_keys(graphs, sample_size: int = 5):
    keys = {"H": set(), "C": set(), "Others": set()}
    for g in graphs[:sample_size]:
        for _, attrs in g.nodes(data=True):
            ntype = node_type_for(attrs)
            keys[ntype].update(attrs.keys())
    return {k: sorted(v) for k, v in keys.items()}


def collect_edge_keys(graphs, sample_size: int = 5):
    keys = set()
    for g in graphs[:sample_size]:
        for _, _, attrs in g.edges(data=True):
            keys.update(attrs.keys())
    return sorted(keys)


def build_node_order(nx_g):
    order = {"H": [], "C": [], "Others": []}
    for node, attrs in nx_g.nodes(data=True):
        order[node_type_for(attrs)].append(node)
    return order


def build_edge_order(nx_g):
    edge_order = {}
    for u, v, _ in nx_g.edges(data=True):
        u_type = node_type_for(nx_g.nodes[u])
        v_type = node_type_for(nx_g.nodes[v])
        forward = (u_type, "bond", v_type)
        reverse = (v_type, "bond", u_type)
        edge_order.setdefault(forward, []).append((u, v))
        edge_order.setdefault(reverse, []).append((v, u))
    return edge_order


def graph_positions(nx_g):
    pos = {}
    for n, attrs in nx_g.nodes(data=True):
        p = attrs.get("pos")
        if p is not None and len(p) >= 2:
            pos[n] = (float(p[0]), float(p[1]))
    if len(pos) != nx_g.number_of_nodes():
        pos = nx.spring_layout(nx_g, seed=0)
    return pos


def _prefixed_int(text: str, prefix: str) -> Optional[int]:
    if text.startswith(prefix):
        try:
            return int(text[len(prefix):])
        except ValueError:
            return None
    return None


def parse_expl_file_metadata(path: Path):
    stem = path.stem
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    method = parts[0]
    node_type = parts[1] if len(parts) > 1 else None
    node_idx = _prefixed_int(parts[2], "n") if len(parts) > 2 else None
    graph_idx = _prefixed_int(parts[3], "g") if len(parts) > 3 else None
    return {
        "path": path,
        "method": method,
        "node_type": node_type,
        "node_idx": node_idx,
        "graph_idx": graph_idx,
    }


In [ ]:
def _device_from_str(device_str: str):
    if not TORCH_AVAILABLE:
        return None
    if device_str and device_str != "auto":
        return torch.device(device_str)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@lru_cache(maxsize=1)
def cached_dataset(data_path: str, config_path: str, norm_stats_path: str, edge_stats_path: str):
    if build_dataset is None:
        raise RuntimeError("explainer_utils nicht verfügbar. Bitte Torch/torch-geometric installieren.")
    config = load_config(config_path)
    norm_stats, edge_stats = load_stats(norm_stats_path, edge_stats_path)
    return build_dataset(data_path, config, norm_stats=norm_stats, edge_stats=edge_stats)


@lru_cache(maxsize=2)
def cached_model(model_path: str, config_path: str, device_str: str):
    if load_trained_model is None:
        raise RuntimeError("explainer_utils nicht verfügbar. Bitte Torch/torch-geometric installieren.")
    device = _device_from_str(device_str)
    config = load_config(config_path)
    model = load_trained_model(model_path, config, device)
    return model, device


def predict_graph(
    data_path: Path,
    graph_idx: int,
    model_path: Path | None = None,
    config_path: Path | None = None,
    norm_stats_path: Path | None = None,
    edge_stats_path: Path | None = None,
    device_str: str = "auto",
):
    model_path = model_path or (MODELS_DIR / "SAGEConv_best_model.pt")
    config_path = config_path or (MODELS_DIR / "config.pkl")
    norm_stats_path = norm_stats_path or (MODELS_DIR / "norm_stats.pkl")
    edge_stats_path = edge_stats_path or (MODELS_DIR / "edge_stats.pkl")

    dataset = cached_dataset(str(data_path), str(config_path), str(norm_stats_path), str(edge_stats_path))
    model, device = cached_model(str(model_path), str(config_path), device_str)

    data = dataset[graph_idx].to(device)
    x_dict, edge_index_dict, edge_attr_dict, _ = heterodata_to_dicts(data)
    with torch.no_grad():
        out = model(x_dict, edge_index_dict, edge_attr_dict)

    pred_dict: Dict[str, list[float]] = {}
    for ntype, tensor in out.items():
        if tensor is None:
            continue
        pred_dict[ntype] = tensor.detach().cpu().view(-1).tolist()
    return pred_dict


def map_predictions_to_nodes(nx_g, pred_dict):
    node_order = build_node_order(nx_g)
    node_preds = {}
    for ntype, nodes in node_order.items():
        values = pred_dict.get(ntype, [])
        for idx, node in enumerate(nodes):
            if idx < len(values):
                node_preds[node] = float(values[idx])
    return node_preds


def load_explainer_importances(expl_path: Path):
    if not TORCH_AVAILABLE:
        raise RuntimeError("PyTorch fehlt - Importances können nicht geladen werden.")
    explanation = torch.load(expl_path, map_location="cpu")
    node_masks = getattr(explanation, "node_mask_dict", {}) or {}
    edge_masks = getattr(explanation, "edge_mask_dict", {}) or {}
    node_masks = {k: v.detach().cpu() for k, v in node_masks.items() if v is not None}
    edge_masks = {k: v.detach().cpu() for k, v in edge_masks.items() if v is not None}
    return node_masks, edge_masks


def aggregate_node_importance(node_masks, node_order, reduction: str = "mean"):
    scores = {}
    for ntype, nodes in node_order.items():
        mask = node_masks.get(ntype)
        if mask is None:
            continue
        if mask.dim() > 1:
            if reduction == "max":
                vals = mask.abs().max(dim=-1).values
            elif reduction == "sum":
                vals = mask.abs().sum(dim=-1)
            else:
                vals = mask.abs().mean(dim=-1)
        else:
            vals = mask
        vals = vals.view(-1)
        for idx, node in enumerate(nodes):
            if idx < vals.numel():
                scores[node] = float(vals[idx])
    return scores


def aggregate_edge_importance(edge_masks, edge_order):
    scores = {}
    for edge_type, edges in edge_order.items():
        mask = edge_masks.get(edge_type)
        if mask is None:
            continue
        vals = mask.view(-1)
        for idx, (u, v) in enumerate(edges):
            if idx >= len(vals):
                break
            key = tuple(sorted((u, v)))
            val = float(vals[idx])
            current = scores.get(key, 0.0)
            if abs(val) > abs(current):
                scores[key] = val
    return scores


In [ ]:
def format_node_label(node_id, attrs, selected_keys, predictions, node_importance):
    selected_keys = list(selected_keys) if selected_keys else []
    lines = [f"{attrs.get('element', '?')}{node_id}"]
    if node_id in predictions:
        lines.append(f"pred={predictions[node_id]:.3f}")
    if node_id in node_importance:
        lines.append(f"imp={node_importance[node_id]:.3f}")
    for key in selected_keys:
        if key in attrs:
            val = attrs[key]
            if isinstance(val, float):
                lines.append(f"{key}={val:.3f}")
            else:
                lines.append(f"{key}={val}")
    return "
".join(lines)


def build_edge_labels(nx_g, edge_keys, edge_importance, min_edge_importance):
    labels = {}
    edge_keys = list(edge_keys) if edge_keys else []
    for u, v, attrs in nx_g.edges(data=True):
        key = tuple(sorted((u, v)))
        parts = []
        for e_key in edge_keys:
            if e_key in attrs:
                val = attrs[e_key]
                if isinstance(val, float):
                    parts.append(f"{e_key}={val:.3f}")
                else:
                    parts.append(f"{e_key}={val}")
        if edge_importance and key in edge_importance and abs(edge_importance[key]) >= min_edge_importance:
            parts.append(f"imp={edge_importance[key]:.3f}")
        if parts:
            labels[(u, v)] = " | ".join(parts)
    return labels


def visualize_graph(
    nx_g,
    node_keys=None,
    edge_keys=None,
    predictions=None,
    node_imp=None,
    edge_imp=None,
    edge_imp_min: float = 0.0,
    figsize=(9, 7),
):
    node_keys = list(node_keys) if node_keys else []
    edge_keys = list(edge_keys) if edge_keys else []
    predictions = predictions or {}
    node_imp = node_imp or {}
    edge_imp = edge_imp or {}

    pos = graph_positions(nx_g)
    color_map = {"H": "#2b8cbe", "C": "#31a354", "Others": "#756bb1"}
    node_colors = [color_map.get(node_type_for(attrs), "#636363") for _, attrs in nx_g.nodes(data=True)]

    widths = []
    for u, v in nx_g.edges():
        imp_val = edge_imp.get(tuple(sorted((u, v)))) if edge_imp else None
        if imp_val is None or abs(imp_val) < edge_imp_min:
            widths.append(1.0)
        else:
            widths.append(1.0 + 4.0 * abs(imp_val))

    fig, ax = plt.subplots(figsize=figsize)
    nx.draw_networkx_edges(nx_g, pos, width=widths, alpha=0.9, edge_color="#444", ax=ax)
    nx.draw_networkx_nodes(
        nx_g,
        pos,
        node_color=node_colors,
        node_size=600,
        edgecolors="#222",
        linewidths=1.0,
        ax=ax,
    )

    labels = {
        node: format_node_label(node, attrs, node_keys, predictions, node_imp)
        for node, attrs in nx_g.nodes(data=True)
    }
    nx.draw_networkx_labels(nx_g, pos, labels=labels, font_size=8, ax=ax)

    edge_labels = build_edge_labels(nx_g, edge_keys, edge_imp, edge_imp_min)
    if edge_labels:
        nx.draw_networkx_edge_labels(nx_g, pos, edge_labels=edge_labels, font_size=7, ax=ax)

    ax.set_axis_off()
    ax.set_title("Molekül-Graph")
    plt.show()


In [ ]:
data_files = list_pickles()
default_data = None
if "all_graphs_with_length.pkl" in data_files:
    default_data = "all_graphs_with_length.pkl"
elif data_files:
    default_data = data_files[0]

if default_data is None:
    raise FileNotFoundError("Keine pkl-Dateien unter data/ gefunden.")

initial_graphs = load_graphs_from_pickle(DATA_DIR / default_data)

# Basis-Widgets
data_file_widget = widgets.Dropdown(options=data_files, value=default_data, description="pkl:")
graph_idx_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=max(0, len(initial_graphs) - 1),
    step=1,
    description="Graph idx",
    continuous_update=False,
)
node_key_options = sorted(set().union(*collect_node_keys(initial_graphs).values()))
node_keys_widget = widgets.SelectMultiple(
    options=node_key_options,
    value=(),
    rows=8,
    description="Node-Keys",
    layout=widgets.Layout(width="320px"),
)
edge_keys_widget = widgets.SelectMultiple(
    options=collect_edge_keys(initial_graphs),
    value=(),
    rows=5,
    description="Edge-Keys",
    layout=widgets.Layout(width="320px"),
)

# Prediction-Widgets
show_predictions_widget = widgets.Checkbox(value=False, description="Predictions einblenden")
model_files = [p.name for p in MODELS_DIR.glob("*.pt")] if MODELS_DIR.exists() else []
model_file_widget = widgets.Dropdown(
    options=model_files or ["SAGEConv_best_model.pt"],
    value=model_files[0] if model_files else "SAGEConv_best_model.pt",
    description="Modell",
    disabled=not show_predictions_widget.value,
)

# Explainer-Widgets
expl_pt_files = [p for p in (EXPLAIN_DIR.glob("*.pt") if EXPLAIN_DIR.exists() else [])]
expl_meta_all = [m for p in expl_pt_files if (m := parse_expl_file_metadata(p))]

def methods_for_graph(graph_idx: int):
    methods = {m["method"] for m in expl_meta_all if m.get("graph_idx") == graph_idx}
    return sorted(methods) if methods else ["gnnexplainer"]

explain_toggle_widget = widgets.Checkbox(value=False, description="Explainer einblenden")
method_options = methods_for_graph(graph_idx_widget.value)
expl_method_widget = widgets.Dropdown(
    options=method_options,
    value="gnnexplainer" if "gnnexplainer" in method_options else method_options[0],
    description="Methode",
)
initial_max_node = max(initial_graphs[0].nodes()) if initial_graphs and initial_graphs[0].nodes else 0
expl_node_idx_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=initial_max_node,
    step=1,
    description="atom_idx",
    continuous_update=False,
)
node_reduce_widget = widgets.Dropdown(
    options=[("mean", "mean"), ("max", "max"), ("sum", "sum")],
    value="mean",
    description="Node-Reduktion",
)
edge_imp_threshold_widget = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=2.0,
    step=0.05,
    description="min |edge imp|",
    continuous_update=False,
)

info_html = widgets.HTML()
out = widgets.Output()


def update_after_data(change=None):
    graphs = load_graphs_from_pickle(DATA_DIR / data_file_widget.value)
    graph_idx_widget.max = max(0, len(graphs) - 1)
    if graph_idx_widget.value > graph_idx_widget.max:
        graph_idx_widget.value = graph_idx_widget.max
    node_keys_widget.options = sorted(set().union(*collect_node_keys(graphs).values()))
    edge_keys_widget.options = collect_edge_keys(graphs)
    update_expl_controls()


def update_expl_controls(change=None):
    graphs = load_graphs_from_pickle(DATA_DIR / data_file_widget.value)
    if not graphs:
        return
    g = graphs[min(graph_idx_widget.value, len(graphs) - 1)]
    max_node_id = max(g.nodes()) if g.nodes else 0
    expl_node_idx_widget.max = max_node_id
    if expl_node_idx_widget.value > max_node_id:
        expl_node_idx_widget.value = max_node_id
    methods = methods_for_graph(graph_idx_widget.value)
    expl_method_widget.options = methods
    if expl_method_widget.value not in methods:
        expl_method_widget.value = methods[0]


def toggle_model_dropdown(change=None):
    model_file_widget.disabled = not show_predictions_widget.value


def pick_expl_file(graph_idx: int, atom_idx: int, nx_g):
    node_attrs = nx_g.nodes.get(atom_idx)
    node_type = node_type_for(node_attrs) if node_attrs else None
    candidates = [m for m in expl_meta_all if m.get("graph_idx") == graph_idx and m.get("method") == expl_method_widget.value]
    if atom_idx is not None:
        candidates = [m for m in candidates if m.get("node_idx") == atom_idx]
    if node_type:
        typed = [m for m in candidates if m.get("node_type") == node_type]
        candidates = typed or candidates
    if not candidates:
        return None
    candidates.sort(key=lambda m: m["path"].name, reverse=True)
    return candidates[0]


def render_graph():
    data_path = DATA_DIR / data_file_widget.value
    graphs = load_graphs_from_pickle(data_path)
    if graph_idx_widget.value >= len(graphs):
        print(f"Graph-Index {graph_idx_widget.value} liegt außerhalb des Bereichs.")
        return
    g = graphs[graph_idx_widget.value]

    info_html.value = (
        f"<b>{data_file_widget.value}</b>: {len(graphs)} Graphen | "
        f"aktuell Graph #{graph_idx_widget.value} mit {g.number_of_nodes()} Knoten / {g.number_of_edges()} Kanten."
    )

    predictions = {}
    if show_predictions_widget.value:
        if TORCH_AVAILABLE and build_dataset is not None:
            try:
                pred_dict = predict_graph(data_path, graph_idx_widget.value, model_path=MODELS_DIR / model_file_widget.value)
                predictions = map_predictions_to_nodes(g, pred_dict)
            except Exception as exc:  # noqa: BLE001
                print(f"Vorhersage fehlgeschlagen: {exc}")
        else:
            print("Torch/explainer_utils fehlen - Predictions werden übersprungen.")

    node_imp = {}
    edge_imp = {}
    if explain_toggle_widget.value:
        if not TORCH_AVAILABLE:
            print("Torch fehlt - Importances werden nicht geladen.")
        else:
            expl_meta = pick_expl_file(graph_idx_widget.value, expl_node_idx_widget.value, g)
            if expl_meta is None:
                print("Keine passende Explanation-Datei gefunden (Methode/Graph/atom_idx).")
            else:
                try:
                    node_masks, edge_masks = load_explainer_importances(expl_meta["path"])
                    node_imp = aggregate_node_importance(node_masks, build_node_order(g), reduction=node_reduce_widget.value)
                    edge_imp = aggregate_edge_importance(edge_masks, build_edge_order(g))
                except Exception as exc:  # noqa: BLE001
                    print(f"Explainer-Laden fehlgeschlagen: {exc}")

    visualize_graph(
        g,
        node_keys=list(node_keys_widget.value),
        edge_keys=list(edge_keys_widget.value),
        predictions=predictions,
        node_imp=node_imp,
        edge_imp=edge_imp,
        edge_imp_min=edge_imp_threshold_widget.value,
    )


def refresh_plot(change=None):
    with out:
        clear_output(wait=True)
        render_graph()


# Observers
for widget in [
    data_file_widget,
    graph_idx_widget,
    node_keys_widget,
    edge_keys_widget,
    show_predictions_widget,
    model_file_widget,
    node_reduce_widget,
    edge_imp_threshold_widget,
    explain_toggle_widget,
    expl_method_widget,
    expl_node_idx_widget,
]:
    widget.observe(refresh_plot, names="value")

data_file_widget.observe(update_after_data, names="value")
graph_idx_widget.observe(update_expl_controls, names="value")
show_predictions_widget.observe(toggle_model_dropdown, names="value")

controls_left = widgets.VBox([data_file_widget, graph_idx_widget, node_keys_widget, edge_keys_widget])
controls_right = widgets.VBox([
    show_predictions_widget,
    model_file_widget,
    explain_toggle_widget,
    expl_method_widget,
    expl_node_idx_widget,
    node_reduce_widget,
    edge_imp_threshold_widget,
])
controls = widgets.HBox([controls_left, controls_right])

update_after_data()
refresh_plot()

display(info_html, controls, out)
